In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
print("Hello")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                              roc_auc_score, roc_curve, precision_recall_curve,
                              average_precision_score, f1_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib
import os

# Plot style
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 12

print("✅ All libraries imported successfully")
print(f"XGBoost version: {xgb.__version__}")

In [ ]:
# Kaggle paths
TRAIN_TRANSACTION = '/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv'
TRAIN_IDENTITY    = '/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv'

print("Loading transaction data...")
df_trans = pd.read_csv(TRAIN_TRANSACTION)
print(f"✅ Transactions loaded: {df_trans.shape}")

print("Loading identity data...")
df_id = pd.read_csv(TRAIN_IDENTITY)
print(f"✅ Identity loaded:     {df_id.shape}")

# Merge on TransactionID
df = df_trans.merge(df_id, on='TransactionID', how='left')
print(f"✅ Merged dataset:      {df.shape}")

# Basic overview
print(f"\n{'='*50}")
print(f"DATASET OVERVIEW")
print(f"{'='*50}")
print(f"Total transactions : {len(df):,}")
print(f"Fraud transactions : {df['isFraud'].sum():,}")
print(f"Fraud rate         : {df['isFraud'].mean()*100:.2f}%")
print(f"Date range (days)  : {df['TransactionDT'].max() - df['TransactionDT'].min()}")
print(f"Memory usage       : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Class distribution
ax1 = fig.add_subplot(gs[0, 0])
counts = df['isFraud'].value_counts()
bars = ax1.bar(['Legitimate', 'Fraud'], counts.values, 
                color=['#3B8BD4', '#E24B4A'], edgecolor='white', linewidth=1.5)
ax1.set_title('Class Distribution', fontweight='bold', fontsize=13)
ax1.set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
             f'{val:,}', ha='center', fontweight='bold')

# 2. Transaction amount distribution
ax2 = fig.add_subplot(gs[0, 1])
df[df['isFraud']==0]['TransactionAmt'].clip(upper=1000).hist(
    bins=50, ax=ax2, alpha=0.7, color='#3B8BD4', label='Legitimate')
df[df['isFraud']==1]['TransactionAmt'].clip(upper=1000).hist(
    bins=50, ax=ax2, alpha=0.7, color='#E24B4A', label='Fraud')
ax2.set_title('Transaction Amount Distribution', fontweight='bold', fontsize=13)
ax2.set_xlabel('Amount (capped at $1000)')
ax2.legend()

# 3. Fraud rate by ProductCD
ax3 = fig.add_subplot(gs[0, 2])
fraud_by_product = df.groupby('ProductCD')['isFraud'].mean().sort_values(ascending=False)
fraud_by_product.plot(kind='bar', ax=ax3, color='#E8593C', edgecolor='white')
ax3.set_title('Fraud Rate by Product Code', fontweight='bold', fontsize=13)
ax3.set_xlabel('Product Code')
ax3.set_ylabel('Fraud Rate')
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)

# 4. Transaction amount — fraud vs legit (box)
ax4 = fig.add_subplot(gs[1, 0])
data_box = [df[df['isFraud']==0]['TransactionAmt'].clip(upper=500).dropna(),
            df[df['isFraud']==1]['TransactionAmt'].clip(upper=500).dropna()]
bp = ax4.boxplot(data_box, patch_artist=True, labels=['Legitimate','Fraud'])
bp['boxes'][0].set_facecolor('#3B8BD4')
bp['boxes'][1].set_facecolor('#E24B4A')
ax4.set_title('Transaction Amount Boxplot', fontweight='bold', fontsize=13)
ax4.set_ylabel('Amount (capped $500)')

# 5. Missing value heatmap (top 20 cols)
ax5 = fig.add_subplot(gs[1, 1])
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False).head(20)
missing.plot(kind='barh', ax=ax5, color='#BA7517')
ax5.set_title('Top 20 Columns: % Missing', fontweight='bold', fontsize=13)
ax5.set_xlabel('% Missing')

# 6. Fraud by card type
ax6 = fig.add_subplot(gs[1, 2])
if 'card4' in df.columns:
    fraud_card = df.groupby('card4')['isFraud'].mean().sort_values(ascending=False)
    fraud_card.plot(kind='bar', ax=ax6, color='#534AB7', edgecolor='white')
    ax6.set_title('Fraud Rate by Card Network', fontweight='bold', fontsize=13)
    ax6.set_xticklabels(ax6.get_xticklabels(), rotation=30)

plt.suptitle('IEEE-CIS Fraud Detection — Exploratory Data Analysis', 
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA plot saved as eda_overview.png")

In [ ]:
def engineer_features(df):
    """
    Build 25+ meaningful features from raw transaction data.
    This is the core of what separates a real ML project from a tutorial.
    """
    df = df.copy()
    
    print("Engineering features...")

    # ── 1. Time-based features ──────────────────────────────────────────
    # TransactionDT is seconds offset from some reference point
    df['hour']       = (df['TransactionDT'] // 3600) % 24
    df['day_of_week'] = (df['TransactionDT'] // (3600 * 24)) % 7
    df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_night']    = df['hour'].isin(range(0, 6)).astype(int)

    # ── 2. Transaction amount features ─────────────────────────────────
    df['amt_log']        = np.log1p(df['TransactionAmt'])
    df['amt_cents']      = df['TransactionAmt'] % 1           # .99 pricing pattern
    df['amt_is_round']   = (df['TransactionAmt'] % 10 == 0).astype(int)
    df['amt_bin']        = pd.cut(df['TransactionAmt'], 
                                   bins=[0,10,50,200,500,10000],
                                   labels=[0,1,2,3,4]).astype(float)

    # ── 3. Card-based aggregates (velocity features) ────────────────────
    # How does this transaction compare to this card's history?
    for col in ['card1', 'card2']:
        grp = df.groupby(col)['TransactionAmt']
        df[f'{col}_amt_mean'] = df[col].map(grp.mean())
        df[f'{col}_amt_std']  = df[col].map(grp.std()).fillna(0)
        df[f'{col}_txn_count']= df[col].map(grp.count())
        df[f'{col}_amt_zscore']= (
            (df['TransactionAmt'] - df[f'{col}_amt_mean']) /
            (df[f'{col}_amt_std'] + 1e-9)
        )

    # ── 4. Email domain features ────────────────────────────────────────
    for col in ['P_emaildomain', 'R_emaildomain']:
        if col in df.columns:
            df[f'{col}_domain'] = df[col].fillna('unknown').apply(
                lambda x: x.split('.')[-1] if '.' in str(x) else x)
            # Flag risky domains
            risky = ['gmail', 'yahoo', 'hotmail', 'outlook']
            df[f'{col}_is_free'] = df[col].fillna('').apply(
                lambda x: 1 if any(r in str(x) for r in risky) else 0)

    # ── 5. Address match feature ────────────────────────────────────────
    if 'addr1' in df.columns and 'addr2' in df.columns:
        df['addr_match'] = (df['addr1'] == df['addr2']).astype(int)
        df['addr1_null'] = df['addr1'].isnull().astype(int)

    # ── 6. V-feature PCA proxies (V1-V339) ─────────────────────────────
    # V features are anonymised — summarise them rather than use raw
    v_cols = [c for c in df.columns if c.startswith('V')]
    if len(v_cols) > 0:
        v_subset = df[v_cols].fillna(-999)
        df['v_mean']    = v_subset.mean(axis=1)
        df['v_std']     = v_subset.std(axis=1)
        df['v_sum']     = v_subset.sum(axis=1)
        df['v_null_count'] = df[v_cols].isnull().sum(axis=1)

    # ── 7. C-features (count features) ─────────────────────────────────
    c_cols = [c for c in df.columns if c.startswith('C')]
    if len(c_cols) > 0:
        df['c_mean'] = df[c_cols].fillna(0).mean(axis=1)
        df['c_sum']  = df[c_cols].fillna(0).sum(axis=1)

    # ── 8. Merchant risk score (encode ProductCD) ───────────────────────
    if 'ProductCD' in df.columns:
        fraud_rate = df.groupby('ProductCD')['isFraud'].mean()
        df['product_fraud_rate'] = df['ProductCD'].map(fraud_rate)

    print(f"✅ Feature engineering complete")
    print(f"   Original columns : {df_trans.shape[1]}")
    print(f"   After engineering: {df.shape[1]}")
    return df

df_feat = engineer_features(df)

In [ ]:
def prepare_data(df):
    # Columns to drop
    drop_cols = ['TransactionID', 'TransactionDT', 'isFraud',
                 'P_emaildomain', 'R_emaildomain',  # replaced by domain features
                 'id_12','id_15','id_16','id_23','id_27','id_28','id_29',
                 'id_30','id_31','id_32','id_33','id_34','id_35','id_36',
                 'id_37','id_38']
    drop_cols = [c for c in drop_cols if c in df.columns]
    
    X = df.drop(columns=drop_cols)
    y = df['isFraud']
    
    # Label encode remaining categoricals
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    print(f"Label encoding {len(cat_cols)} categorical columns...")
    le = LabelEncoder()
    for col in cat_cols:
        X[col] = X[col].fillna('missing')
        X[col] = le.fit_transform(X[col].astype(str))
    
    # Fill remaining nulls with median
    X = X.fillna(X.median())
    
    print(f"✅ Features ready: {X.shape[1]} columns")
    print(f"   Positive (fraud) samples: {y.sum():,} ({y.mean()*100:.2f}%)")
    return X, y

X, y = prepare_data(df_feat)

# Train/test split — stratified to preserve fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTrain size : {X_train.shape[0]:,}")
print(f"Test size  : {X_test.shape[0]:,}")
print(f"Train fraud: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Test fraud : {y_test.sum():,} ({y_test.mean()*100:.2f}%)")

In [ ]:
print("Applying SMOTE to balance training data...")
print(f"Before SMOTE — Fraud: {y_train.sum():,}, Legit: {(y_train==0).sum():,}")

# SMOTE creates synthetic fraud samples so model learns the minority class
smote = SMOTE(random_state=42, sampling_strategy=0.3)  
# 0.3 = for every 1 fraud, we want 0.3x the legit count (not full 1:1, avoids overfitting)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"After SMOTE  — Fraud: {y_train_sm.sum():,}, Legit: {(y_train_sm==0).sum():,}")
print(f"New fraud rate: {y_train_sm.mean()*100:.1f}%")
print("✅ SMOTE complete — data is now balanced for training")

In [ ]:
results = {}

# ── Model 1: Logistic Regression (baseline) ─────────────────────────────
print("Training Logistic Regression (baseline)...")
lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train_sm, y_train_sm)
lr_proba = lr.predict_proba(X_test)[:, 1]
results['Logistic Regression'] = {
    'proba': lr_proba,
    'auc': roc_auc_score(y_test, lr_proba),
    'model': lr
}
print(f"  ✅ LR AUC-ROC: {results['Logistic Regression']['auc']:.4f}")

# # ── Model 2: XGBoost (main model) ───────────────────────────────────────
# print("\nTraining XGBoost (main model)...")
# xgb_model = xgb.XGBClassifier(
#     n_estimators=300,
#     max_depth=6,
#     learning_rate=0.05,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     scale_pos_weight=1,      # SMOTE already handled balance
#     use_label_encoder=False,
#     eval_metric='auc',
#     random_state=42,
#     n_jobs=-1,
#     tree_method='hist'       # fast on Kaggle GPU
# )
# xgb_model.fit(
#     X_train_sm, y_train_sm,
#     eval_set=[(X_test, y_test)],
#     verbose=50
# )
# xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
# results['XGBoost'] = {
#     'proba': xgb_proba,
#     'auc': roc_auc_score(y_test, xgb_proba),
#     'model': xgb_model
# }
# print(f"\n  ✅ XGBoost AUC-ROC: {results['XGBoost']['auc']:.4f}")

print("Training XGBoost v2 (500 trees)...")
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    scale_pos_weight=1,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=30      # stops automatically if no improvement
)

xgb_model.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test, y_test)],
    verbose=50
)

xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, xgb_proba)
results['XGBoost'] = {'proba': xgb_proba, 'auc': auc, 'model': xgb_model}

print(f"\n✅ XGBoost v2 AUC-ROC: {auc:.4f}")
print(f"   Best iteration    : {xgb_model.best_iteration}")

In [ ]:
# Pick best threshold for XGBoost
def find_best_threshold(y_true, y_proba):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
    best_idx   = np.argmax(f1_scores[:-1])
    return thresholds[best_idx], f1_scores[best_idx]

best_thresh, best_f1 = find_best_threshold(y_test, xgb_proba)
y_pred_best = (xgb_proba >= best_thresh).astype(int)

print(f"Best threshold : {best_thresh:.3f}")
print(f"Best F1 score  : {best_f1:.4f}")
print(f"\nClassification Report (XGBoost @ best threshold):")
print(classification_report(y_test, y_pred_best, 
                             target_names=['Legitimate', 'Fraud']))

# ── Plots ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. ROC curves
ax = axes[0, 0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC={res['auc']:.4f})")
ax.plot([0,1],[0,1],'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison', fontweight='bold')
ax.legend()

# 2. Precision-Recall curve (more meaningful for imbalanced data)
ax = axes[0, 1]
prec, rec, _ = precision_recall_curve(y_test, xgb_proba)
ap = average_precision_score(y_test, xgb_proba)
ax.plot(rec, prec, color='#E24B4A', lw=2, label=f'XGBoost (AP={ap:.4f})')
ax.axhline(y=y_test.mean(), color='gray', linestyle='--', label='Random baseline')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontweight='bold')
ax.legend()

# 3. Confusion matrix
ax = axes[1, 0]
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Legit','Fraud'], yticklabels=['Legit','Fraud'])
ax.set_title(f'Confusion Matrix (threshold={best_thresh:.2f})', fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')

# 4. Feature importance (top 20)
ax = axes[1, 1]
feat_imp = pd.Series(
    xgb_model.feature_importances_, index=X_train.columns
).sort_values(ascending=False).head(20)
feat_imp.plot(kind='barh', ax=ax, color='#534AB7')
ax.invert_yaxis()
ax.set_title('Top 20 Feature Importances (XGBoost)', fontweight='bold')
ax.set_xlabel('Importance score')

plt.suptitle('IEEE-CIS Fraud Detection — Model Evaluation', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('model_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Results plot saved as model_results.png")

In [ ]:
import os
os.makedirs('model', exist_ok=True)

# Save XGBoost model
joblib.dump(xgb_model, 'model/fraud_model.pkl')
joblib.dump(X_train.columns.tolist(), 'model/feature_names.pkl')
joblib.dump(best_thresh, 'model/best_threshold.pkl')

print("✅ Model artifacts saved:")
print("   model/fraud_model.pkl")
print("   model/feature_names.pkl")
print("   model/best_threshold.pkl")

# Final summary — this is what goes in your README
print(f"""
{'='*55}
PROJECT RESULTS SUMMARY
{'='*55}
Dataset       : IEEE-CIS Fraud Detection (Kaggle)
Transactions  : {len(df):,} total | {df['isFraud'].sum():,} fraud ({df['isFraud'].mean()*100:.2f}%)
Features built: {X.shape[1]} engineered features
SMOTE         : Applied (sampling_strategy=0.3)

MODEL PERFORMANCE
-----------------
Logistic Regression AUC-ROC : {results['Logistic Regression']['auc']:.4f}
XGBoost AUC-ROC             : {results['XGBoost']['auc']:.4f}
XGBoost F1 (best threshold) : {best_f1:.4f}
Best classification threshold: {best_thresh:.3f}
{'='*55}
""")

In [ ]:
def predict_fraud(transaction: dict, model, feature_names, threshold) -> dict:
    """
    This exact function will become your FastAPI endpoint.
    Input: a single transaction as a dict
    Output: fraud probability + decision
    """
    row = pd.DataFrame([transaction])
    
    # Align to training features
    for col in feature_names:
        if col not in row.columns:
            row[col] = 0
    row = row[feature_names].fillna(0)
    
    proba = model.predict_proba(row)[0][1]
    decision = 'FRAUD' if proba >= threshold else 'LEGITIMATE'
    
    return {
        'fraud_probability': round(float(proba), 4),
        'decision': decision,
        'threshold_used': round(float(threshold), 3),
        'risk_level': 'HIGH' if proba > 0.7 else 'MEDIUM' if proba > 0.4 else 'LOW'
    }

# Test with a sample transaction
sample = X_test.iloc[0].to_dict()
result = predict_fraud(sample, xgb_model, X_train.columns.tolist(), best_thresh)

print("Sample prediction:")
print(f"  Fraud probability : {result['fraud_probability']}")
print(f"  Decision          : {result['decision']}")
print(f"  Risk level        : {result['risk_level']}")
print(f"\nActual label: {'FRAUD' if y_test.iloc[0]==1 else 'LEGITIMATE'}")
print("\n✅ Prediction function works — ready to wrap in FastAPI")